# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [117]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [118]:
# using requests to save local copy of the pdf

import requests
url = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
pdf_report = requests.get(url)
with open ('ai_report_2025.pdf', 'wb') as f:
    f.write(pdf_report.content)



In [119]:
# using pypdf drawing from code provided on LangChain documentation

import pypdf
from langchain_core.documents import Document

reader = pypdf.PdfReader('ai_report_2025.pdf')   #reading the pdf file

docs = [
    Document(
        page_content=page.extract_text() or "",
        metadata={"page": i}
    )
    for i, page in enumerate(reader.pages)    #looping through each page to read the text
]


In [120]:
# joining pages together

full_pdf_text =  ""
for page in docs:
    full_pdf_text += page.page_content + "\n"


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [149]:
import sys
sys.path.append('../05_src/')

from utils.clients import get_client
import os
from IPython.display import display, Markdown

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client(use_gateway=True)


# OpenAI Gateway Setup Notes
# AuthenticationError (401) was raised during generation task.
# Troubleshooting with client.base_url showed that requests went to OpenAI directly instead of the gateway.
# I set get_client(use_gateway=True) to override the default and force it to use course gateway instead.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field
class PdfAnalysis(BaseModel):
    Author: str=Field(description="The author of the article")
    Title: str=Field(description="The title of the article")
    Relevance: str=Field(description="max one paragraph on why article is relevant for an AI professional in their professional development")
    Summary: str=Field(description="A concise summary no longer than 1000 tokens")
    Tone: str=Field(description="The tone used to produce the summary")
    InputTokens: Optional[int]=Field(description="The number of input tokens from response object")
    OutputTokens: Optional[int]=Field(description="The number of output tokens from response object")


# following code provided on structured_outputs jupyter notebook
# setting Optional for Input and Output tokens to allow return of null

In [ ]:
instructions = """
    You are a research assistant.
    Analyze the article and extract structured information.
    
    Return:
    - Author
    - Title
    - Relevance: a statement, maximum one paragraph, explaining why is the article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary
    - InputTokens: number of input tokens from response object
    - OutputTokens: number of input tokens from response object
    """

tone = "Formal Academic Writing"

prompt = f"""
    Analyze the following article.
    <article>
    {full_pdf_text}
    </article>
    
    Write the summary in {tone}.
    
    """


In [162]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": prompt},
    ],
    text_format=PdfAnalysis,
)

result = response.output_parsed

result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

result 

# following code provided on structured_outputs jupyter notebook

PdfAnalysis(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This article is crucial for AI professionals as it highlights the current landscape of generative AI implementations in enterprises, emphasizing the substantial gap between high adoption rates and low transformational impact. Understanding these dynamics will aid AI professionals in developing strategies that not only focus on technology but also organizational design, user experience, and partnership dynamics, all critical for achieving successful AI integration and maximizing ROI.', Summary="The report 'The GenAI Divide', published by MIT NANDA in July 2025, presents findings from a comprehensive research initiative investigating generative AI (GenAI) implementations across organizations. Despite significant investments—estimated between $30-40 billion—95% of enterprises report no meaningful return on their GenAI initiativ

In [ ]:
#displaying result in easy-to-read Markdown format

display(Markdown(f"""
**Author:** {result.Author}

**Title:** {result.Title}

**Relevance:** {result.Relevance}

**Summary:** {result.Summary}

**Tone:** {result.Tone}

**Input Tokens:** {result.InputTokens}

**Output Tokens:** {result.OutputTokens}
"""))


**Author:** MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

**Title:** The GenAI Divide: State of AI in Business 2025

**Relevance:** This article is critical for AI professionals as it delineates the stark contrast between high adoption rates of generative AI technologies and the low transformational outcomes across organizations. Understanding the 'GenAI Divide' provides insights into effective AI implementation, overcoming common pitfalls, and aligning AI strategies with business objectives—key competencies for professionals looking to lead in AI-driven enterprises.

**Summary:** This report presents findings from preliminary research conducted under Project NANDA, examining generative AI (GenAI) implementation across organizations. Despite significant investments ranging from $30 to $40 billion, a staggering 95% of enterprises reported minimal returns on their AI initiatives, revealing a pronounced divide between organizations classified as 'builders' and 'buyers.' High adoption rates, particularly of tools like ChatGPT and Copilot, have not translated into meaningful business transformation, as most deployments yield zero significant impact on profit and loss. Four emergent patterns delineate the divide: limited disruption across sectors, a paradox where larger firms lag in scaling successful pilots, biases in investment favoring visible functions over those yielding high ROI, and the superior performance of external partnerships compared to internal initiatives. Challenges such as inadequate contextual learning and integration with existing workflows impede many AI projects, while 'shadow AI'—the use of personal AI tools among employees—exhibits a more favorable ROI than formal organizational initiatives. Ultimately, organizations that have successfully navigated the GenAI Divide demonstrate a commitment to adaptive, learning-capable systems that integrate deeply into workflows, emphasizing the necessity for enterprises to reassess their strategy towards AI acquisition and implementation in a rapidly evolving technological landscape. The conclusions advocate for a shift in focus from building to buying tailored solutions that meet specific organizational needs, fostering partnerships to enhance AI deployment effectiveness.

**Tone:** Formal Academic Writing

**Input Tokens:** 11052

**Output Tokens:** 408


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
